In [24]:
import numpy as np

In [25]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, iterations=1000):
        self.lr = lr #learning rate
        self.iterations = iterations 
        self.weights = None 
        self.bias = None 

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X, y):
        # Inizializzazione pesi: un peso per ogni feature
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Gradient Descent
        for _ in range(self.iterations):
            # Passaggio 1: Linear Model (z = Xw + b)
            linear_model = np.dot(X, self.weights) + self.bias
            # Passaggio 2: Applica Sigmoide (y_pred)
            y_predicted = self._sigmoid(linear_model)

            # Passaggio 3: Calcolo Gradienti (derivata della Log Loss)
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Passaggio 4: Aggiornamento parametri
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
    
    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_model)
    
    def predict(self, X):
        probs = self.predict_proba(X)
        return [1 if i > 0.5 else 0 for i in probs]

In [26]:
# Input (X): numeri da 1 a 20
X = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 19]).reshape(-1, 1)

# Target (y): 0 se <= 10, 1 se > 10
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])


In [27]:
# Inizializziamo il modello
model = LogisticRegressionScratch(lr=0.1, iterations=1000)

# Vediamo i pesi iniziali (casuali o zero)
print(f"Pesi iniziali: w={model.weights}, b={model.bias}")

model.fit(X, y)

print(f"Pesi finali: w={model.weights[0]:.4f}, b={model.bias:.4f}")

test_data = np.array([2, 18]).reshape(-1, 1)
predictions = model.predict(test_data)

for val, pred in zip(test_data.flatten(), predictions):
    print(f"Numero: {val} -> Classe Predetta: {pred} ({'Corretto!' if (val > 10) == pred else 'Sbagliato!'})")

Pesi iniziali: w=None, b=None
Pesi finali: w=0.5892, b=-5.6032
Numero: 2 -> Classe Predetta: 0 (Corretto!)
Numero: 18 -> Classe Predetta: 1 (Corretto!)


In [28]:
def evaluate_classification(y_true, y_pred_probs, threshold=0.5):
    # Convertiamo le probabilità in classi 0 o 1 in base alla soglia
    y_pred = (y_pred_probs >= threshold).astype(int)
    
    # Elementi della Confusion Matrix
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    # Calcolo Metriche
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        "Confusion Matrix": [[tn, fp], [fn, tp]],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }


In [29]:
def calculate_roc_auc(y_true, y_pred_probs):
    # Testiamo 100 soglie diverse da 1.0 a 0.0
    thresholds = np.linspace(1, 0, 100)
    tpr_list = [] # True Positive Rate
    fpr_list = [] # False Positive Rate
    
    for t in thresholds:
        y_p = (y_pred_probs >= t).astype(int)
        tp = np.sum((y_true == 1) & (y_p == 1))
        fp = np.sum((y_true == 0) & (y_p == 1))
        fn = np.sum((y_true == 1) & (y_p == 0))
        tn = np.sum((y_true == 0) & (y_p == 0))
        
        tpr_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    
    # Calcolo AUC usando la regola del trapezio (area sotto la curva)
    auc = np.trapezoid(tpr_list, fpr_list)
    
    return fpr_list, tpr_list, auc

In [30]:
# Dati reali: [Piccolo, Piccolo, Grande, Grande, Grande]
y_true = np.array([0, 0, 1, 1, 1])

# Probabilità predette dal modello (output della sigmoide)
y_probs = np.array([0.1, 0.4, 0.35, 0.8, 0.9]) 
# Nota: il terzo valore (0.35) è un errore del modello!

# 1. Metriche classiche
metrics = evaluate_classification(y_true, y_probs)
for k, v in metrics.items():
    print(f"{k}: {v}")

# 2. ROC e AUC
fpr, tpr, auc_value = calculate_roc_auc(y_true, y_probs)
print(f"AUC: {auc_value:.4f}")

Confusion Matrix: [[np.int64(2), np.int64(0)], [np.int64(1), np.int64(2)]]
Accuracy: 0.8
Precision: 1.0
Recall: 0.6666666666666666
F1 Score: 0.8
AUC: 0.8333


In [31]:
import torch
import torch.nn as nn

In [32]:
class LogisticRegressionPyTorch(nn.Module):
    def __init__(self, n_features):
        super(LogisticRegressionPyTorch, self).__init__()
        # Definizione dello strato lineare (z = wx + b)
        # PyTorch inizializza pesi e bias automaticamente!
        self.linear = nn.Linear(n_features, 1)
        
    def forward(self, x):
        # Passaggio 1 & 2: Lineare + Sigmoide
        y_predicted = torch.sigmoid(self.linear(x))
        return y_predicted

# --- Preparazione Dati ---
X_np = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 19], dtype=np.float32).reshape(-1, 1)
y_np = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1], dtype=np.float32).reshape(-1, 1)

# Trasformiamo NumPy in Tensor di PyTorch
X = torch.from_numpy(X_np)
y = torch.from_numpy(y_np)

# --- Training Loop ---
model = LogisticRegressionPyTorch(n_features=1)
criterion = nn.BCELoss() # Binary Cross Entropy (la nostra Log Loss)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) # Optimizer

for epoch in range(1000):
    # 1. Forward pass
    y_pred = model(X)
    loss = criterion(y_pred, y)
    
    # 2. Backward pass (Il miracolo dell'Autograd)
    loss.backward()      # Calcola i gradienti automaticamente
    optimizer.step()     # Aggiorna i pesi (w = w - lr * grad)
    optimizer.zero_grad() # Reset dei gradienti per il prossimo giro

    if (epoch+1) % 200 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')

Epoch 200, Loss: 0.2562
Epoch 400, Loss: 0.1966
Epoch 600, Loss: 0.1667
Epoch 800, Loss: 0.1481
Epoch 1000, Loss: 0.1350


In [33]:
# Dopo il loop di training...

model.eval() # Imposta il modello in modalità valutazione
with torch.no_grad(): # Disabilita il calcolo dei gradienti (risparmia memoria)
    # Otteniamo le probabilità dal modello
    y_probs_tensor = model(X) 
    
    # Convertiamo i tensori in array NumPy per le nostre funzioni
    y_true = y.numpy().flatten()
    y_probs = y_probs_tensor.numpy().flatten()

# Ora puoi usare le funzioni che abbiamo scritto prima!
metrics = evaluate_classification(y_true, y_probs)
fpr, tpr, auc_value = calculate_roc_auc(y_true, y_probs)

print("--- Risultati Finali ---")
for k, v in metrics.items():
    print(f"{k}: {v}")
print(f"AUC: {auc_value:.4f}")

--- Risultati Finali ---
Confusion Matrix: [[np.int64(5), np.int64(0)], [np.int64(0), np.int64(5)]]
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
AUC: 1.0000


In [35]:
import jax
import jax.numpy as jnp # Usiamo il numpy di JAX
from jax import grad, jit

In [36]:
# 1. Dati (JAX usa jnp invece di np)
X = jnp.array([[1.], [3.], [5.], [7.], [9.], [11.], [13.], [15.], [17.], [19.]])
y = jnp.array([[0.], [0.], [0.], [0.], [0.], [1.], [1.], [1.], [1.], [1.]])

# 2. Inizializzazione parametri (Dizionario esplicito)
params = {
    "w": jnp.zeros((1, 1)),
    "b": 0.0
}

# 3. Definiamo il Modello (Funzione pura)
def predict(params, x):
    z = jnp.dot(x, params["w"]) + params["b"]
    return jax.nn.sigmoid(z)

# 4. Definiamo la Loss (Binary Cross Entropy)
def loss_fn(params, x, y):
    preds = predict(params, x)
    # Evitiamo log(0) con jnp.clip
    preds = jnp.clip(preds, 1e-7, 1 - 1e-7)
    return -jnp.mean(y * jnp.log(preds) + (1 - y) * jnp.log(1 - preds))

# 5. Trasformazione magica: JAX crea la funzione per il gradiente
grad_fn = grad(loss_fn)

# 6. Training Loop
lr = 0.1
for epoch in range(1000):
    # Calcoliamo i gradienti rispetto ai parametri
    grads = grad_fn(params, X, y)
    
    # Aggiornamento manuale (Funzionale: creiamo nuovi parametri)
    params = {
        "w": params["w"] - lr * grads["w"],
        "b": params["b"] - lr * grads["b"]
    }

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss_fn(params, X, y):.4f}")

Epoch 200, Loss: 0.2810
Epoch 400, Loss: 0.2067
Epoch 600, Loss: 0.1724
Epoch 800, Loss: 0.1518
Epoch 1000, Loss: 0.1378
